# Data Science · Woche 2 · Praktikum

**B.Sc. Wirtschaftsinformatik · Hochschule Ruhr West · Do 08.10.2026, 16:45–18:20**

Heute beschreiben Sie echte Daten mit den Werkzeugen aus der Vorlesung: Häufigkeiten, Mittelwert und Median, Streuung,
Quartile, Boxplots und Anteile als Wahrscheinlichkeiten. Die Daten sind wieder die Bestellungen von Olist.

| Zeit | Was |
|---|---|
| 16:45 | 0 · Setup: eigene Kopie speichern, Daten laden |
| 16:50 | 1 · Häufigkeiten: Wie bezahlen die Kunden? |
| 17:00 | 2 · Lieferdauer: Wie ist sie verteilt? |
| 17:15 | 3 · Kennzahlen: Mitte, Streuung, Quartile |
| 17:35 | 4 · Bundesstaaten vergleichen |
| 17:50 | 5 · Anteile als Wahrscheinlichkeiten |
| 18:00 | 6 · Drei Sätze |
| 18:05 | Besprechung: zwei Teams zeigen ein Ergebnis, dann die Musterlösung (danach in Moodle) |

**So arbeiten Sie:** Zelle ausführen mit `Shift + Enter`. Aufgaben sind mit `# TODO` markiert, dort ersetzen Sie
`None` durch Ihren Code. Darunter prüft ein Check Ihr Ergebnis: ✓ richtig · ✗ prüfen · … noch offen.
Wer früher fertig ist: Am Ende stehen eine Kür mit Simulationen aus der Vorlesung und eine Aufgabe ohne Code.

## 0 · Setup (ca. 5 Minuten)

Wie letzte Woche: **Datei → Kopie in Drive speichern**, dann die Zelle unten ausführen. Sie holt drei Tabellen direkt
aus dem Kurs-Repository auf GitHub: Bestellungen, Kunden und Zahlungen. Erwartet: `orders: 99441 Zeilen`,
`kunden: 99441 Zeilen`, `zahlungen: 103886 Zeilen`.

*Falls das Laden scheitert:* die drei Dateien aus https://github.com/ZhuosHub/Data_Science_HRW_Praktikum (Ordner `data/olist`) herunterladen, in Colab links über das
Ordner-Symbol hochladen und die Zelle erneut ausführen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASIS_URL = 'https://raw.githubusercontent.com/ZhuosHub/Data_Science_HRW_Praktikum/main/data/olist/'


def lade(name, **kw):
    """Sucht die Datei lokal, dann im Kurs-Repository auf GitHub, dann im Arbeitsordner (hochgeladen)."""
    for quelle in ['../data/olist/', BASIS_URL, './']:
        try:
            return pd.read_csv(quelle + name, **kw)
        except (FileNotFoundError, OSError) as fehler:
            if quelle == BASIS_URL:
                print('GitHub nicht erreichbar:', fehler)
    raise FileNotFoundError(f'{name} nicht gefunden – siehe „Falls das Laden scheitert“ über dieser Zelle')


datum = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
orders = lade('olist_orders_dataset.csv', parse_dates=datum)
kunden = lade('olist_customers_dataset.csv')
zahlungen = lade('olist_order_payments_dataset.csv')
print('orders:', len(orders), 'Zeilen · kunden:', len(kunden), 'Zeilen · zahlungen:', len(zahlungen), 'Zeilen')

## 1 · Häufigkeiten: Wie bezahlen die Kunden? (ca. 10 Minuten)

Die Tabelle `zahlungen` hat eine Zeile je Zahlung, die Spalte `payment_type` sagt, wie bezahlt wurde.
- **absolute Häufigkeit:** wie oft jeder Wert vorkommt → `value_counts()`
- **relative Häufigkeit:** derselbe Anteil an allen Zeilen → `value_counts(normalize=True)`
- **Modus:** der häufigste Wert → `mode()[0]`

In [ ]:
# TODO 1.1: absolute und relative Häufigkeiten der Zahlungsarten, dazu der Modus
absolut = None
relativ = None
modus = None

# --- Check ---
if absolut is None or relativ is None or modus is None:
    print('… noch offen')
else:
    print(absolut, '\n')
    print(relativ.round(3), '\n')
    ok = absolut.get('credit_card') == 76795 and abs(relativ.sum() - 1) < 1e-9 and modus == 'credit_card'
    print('✓ richtig' if ok else '✗ prüfen: Kreditkarte 76795-mal, Anteile summieren sich zu 1, Modus credit_card')

In [ ]:
# Balkendiagramm der relativen Häufigkeiten (läuft, sobald 1.1 gelöst ist)
if relativ is not None:
    relativ.plot.bar(color='#4F8F88', title='Zahlungsarten bei Olist (Anteil der Zahlungen)')
    plt.ylabel('Anteil')
    plt.show()

## 2 · Lieferdauer: Wie ist sie verteilt? (ca. 15 Minuten)

Wir nehmen nur zugestellte Bestellungen mit Zustelldatum (wie in Woche 1) und rechnen aus, wie viele Tage zwischen
Bestellung und Zustellung lagen. `.dt.days` macht aus einer Zeitspanne ganze Tage.

In [ ]:
lieferung = orders[(orders['order_status'] == 'delivered') & orders['order_delivered_customer_date'].notna()].copy()

# TODO 2.1: neue Spalte 'tage' = Zustellung minus Bestellung, in ganzen Tagen
lieferung['tage'] = None
tage = lieferung['tage']

# --- Check ---
if tage.isna().all():
    print('… noch offen')
else:
    print('Lieferungen:', len(tage), '· erste Werte:', tage.head(5).tolist())
    print('✓ richtig' if len(tage) == 96470 and abs(tage.mean() - 12.09) < 0.01 else '✗ prüfen: 96470 Lieferungen erwartet')

**2.2 Histogramm.** Ein Histogramm zählt, wie viele Werte in jedes Intervall fallen. Führen Sie die Zelle aus und
vergleichen Sie zwei Klassenbreiten: 20 Klassen und 100 Klassen. Was zeigt die feine Einteilung, was die grobe nicht?

In [ ]:
if not tage.isna().all():
    fig, achsen = plt.subplots(1, 2, figsize=(12, 4))
    for ax, klassen in zip(achsen, [20, 100]):
        tage.plot.hist(bins=klassen, ax=ax, color='#4F8F88', title=f'{klassen} Klassen')
        ax.set_xlabel('Lieferdauer in Tagen')
    plt.show()

**2.3 Ihre Beschreibung (ein Satz):** Ist die Verteilung symmetrisch oder schief? Gibt es Ausreißer?

*Ihre Antwort:*

## 3 · Kennzahlen: Mitte, Streuung, Quartile (ca. 20 Minuten)

Jetzt die Zahlen aus der Vorlesung, an 96 470 Lieferungen. Für die Standardabweichung rechnen Sie einmal **von Hand**
mit der Formel aus der Vorlesung, $s = \sqrt{\tfrac{1}{n}\sum (x_i - \bar x)^2}$, und vergleichen mit pandas.

In [ ]:
# TODO 3.1: die Mitte
mittelwert = None     # .mean()
median = None         # .median()
modus_tage = None     # der häufigste Wert: .mode()[0]

# --- Check ---
if None in (mittelwert, median, modus_tage):
    print('… noch offen')
else:
    print(f'Mittelwert {mittelwert:.2f} · Median {median} · Modus {modus_tage}')
    ok = abs(mittelwert - 12.09) < 0.01 and median == 10 and modus_tage == 7
    print('✓ richtig' if ok else '✗ prüfen: Mittelwert 12.09, Median 10, Modus 7')

In [ ]:
# TODO 3.2: die Streuung
spannweite = None     # größter minus kleinster Wert
varianz = None        # von Hand: Durchschnitt der quadrierten Abweichungen vom Mittelwert
standardabw = None    # von Hand: Wurzel aus der Varianz (np.sqrt)

# --- Check ---
if None in (spannweite, varianz, standardabw):
    print('… noch offen')
else:
    print(f'Spannweite {spannweite} · Varianz {varianz:.2f} · Standardabweichung {standardabw:.4f}')
    print(f'zum Vergleich pandas tage.std(): {tage.std():.4f}  (teilt durch n − 1; bei so vielen Werten fast gleich)')
    ok = spannweite == 209 - 0 and abs(standardabw - 9.55) < 0.01
    print('✓ richtig' if ok else '✗ prüfen: Spannweite 209, Standardabweichung 9,55')

In [ ]:
# TODO 3.3: Quartile und Interquartilsabstand, dazu das 90-%-Quantil
q1 = None             # .quantile(0.25)
q3 = None
iqr = None
p90 = None

# --- Check ---
if None in (q1, q3, iqr, p90):
    print('… noch offen')
else:
    print(f'Q1 {q1} · Q3 {q3} · IQR {iqr} · 90-%-Quantil {p90}')
    print('✓ richtig' if (q1, q3, iqr, p90) == (6, 15, 9, 23) else '✗ prüfen: Q1 6, Q3 15, IQR 9, 90 % 23')

**3.4 `describe()` fasst vieles auf einmal zusammen.** Finden Sie Ihre Zahlen wieder? Lesen Sie die Zeile `50%`, und
schreiben Sie in einem Satz, was das 90-%-Quantil bedeutet: „90 % der Lieferungen dauern höchstens … Tage.“

In [ ]:
tage.describe()

## 4 · Bundesstaaten vergleichen (ca. 15 Minuten)

Wir hängen an jede Lieferung den Bundesstaat der Kundin oder des Kunden an (`customer_state`, Tabelle `kunden`) und
vergleichen die sechs Staaten mit den meisten Lieferungen.

In [ ]:
if 'customer_state' not in lieferung:   # nur einmal anhängen, auch wenn die Zelle zweimal läuft
    lieferung = lieferung.merge(kunden[['customer_id', 'customer_state']], on='customer_id')
top6 = lieferung['customer_state'].value_counts().head(6).index
teil = lieferung[lieferung['customer_state'].isin(top6)]

# TODO 4.1: je Staat Anzahl, Median und Interquartilsabstand der Lieferdauer
# Tipp: teil.groupby('customer_state')['tage'].agg(...); für den IQR eine Funktion lambda x: x.quantile(0.75) - x.quantile(0.25)
vergleich = None

# --- Check ---
if vergleich is None:
    print('… noch offen')
else:
    print(vergleich, '\n')
    ok = vergleich.loc['SP', 'median'] == 7 and vergleich['iqr'].idxmax() == 'RJ'
    print('✓ richtig' if ok else '✗ prüfen: Median in SP 7 Tage, größter IQR in RJ')

In [ ]:
# Boxplots nebeneinander (läuft, sobald 4.1 gelöst ist; Ausreißer ausgeblendet, damit man die Boxen sieht)
if vergleich is not None:
    teil.boxplot(column='tage', by='customer_state', showfliers=False, figsize=(10, 4))
    plt.suptitle('')
    plt.title('Lieferdauer in Tagen, sechs größte Bundesstaaten')
    plt.show()

**4.2 Ihre Beschreibung:** In welchem Staat geht es am schnellsten, wo streut die Lieferdauer am stärksten?

*Ihre Antwort:*

## 5 · Anteile als Wahrscheinlichkeiten (ca. 10 Minuten)

Verspätet heißt wie in Woche 1: Der Liefertag liegt nach dem zugesagten Tag. Der Anteil verspäteter Lieferungen ist
eine geschätzte Wahrscheinlichkeit. Getrennt nach Staat wird daraus eine **bedingte Wahrscheinlichkeit**:
P(verspätet | Staat).

In [ ]:
# TODO 5.1: Spalte 'spaet' (True/False) und der Anteil verspäteter Lieferungen
lieferung['spaet'] = None   # Tipp: ...dt.normalize() > ...dt.normalize()
p_spaet = None              # Anteil der True-Werte: .mean()

# --- Check ---
if p_spaet is None:
    print('… noch offen')
else:
    print(f'P(verspätet) = {p_spaet:.4f}')
    print('✓ richtig' if abs(p_spaet - 0.0677) < 0.0005 else '✗ prüfen: erwartet etwa 6,8 %')

In [ ]:
# TODO 5.2: P(verspätet | Staat) für alle Staaten mit mindestens 1 000 Lieferungen, größter Wert zuerst
je_staat = None   # Tipp: groupby('customer_state')['spaet'].agg(['size', 'mean'])

# --- Check ---
if je_staat is None:
    print('… noch offen')
else:
    print(je_staat.head(5), '\n')
    ok = je_staat.index[0] == 'CE' and abs(je_staat['mean'].iloc[0] - 0.1376) < 0.0005
    print('✓ richtig' if ok else '✗ prüfen: oben steht CE mit etwa 13,8 %')

**5.3 Ihre Einschätzung:** Kommt eine Bestellung aus Ceará (CE) etwa dreimal so oft zu spät wie eine aus São Paulo (SP)?

*Ihre Antwort:*

## 6 · Drei Sätze (ca. 5 Minuten)

Beschreiben Sie eine typische Olist-Lieferung in drei Sätzen: Welche Zahl nehmen Sie für die Mitte, und warum? Wie
stark streut die Lieferdauer? Wo gibt es große Unterschiede?

*Ihre Antwort:*

---
## Kür: Simulationen aus der Vorlesung

Wer fertig ist: Prüfen Sie drei Ergebnisse aus der Vorlesung mit dem Zufallsgenerator nach.
`rng.integers(1, 7, size=...)` würfelt, `rng.integers(0, 365, size=...)` zieht Geburtstage.

In [ ]:
rng = np.random.default_rng(8)

# TODO K1: Mensch ärgere dich nicht – Anteil der Versuche mit mindestens einer Sechs in drei Würfen (100 000 Versuche)
wuerfe = rng.integers(1, 7, size=(100_000, 3))
k1 = None   # Tipp: (wuerfe == 6).any(axis=1).mean()

# --- Check ---
if k1 is None:
    print('… noch offen')
else:
    print(f'simuliert: {k1:.3f} · gerechnet: 1 − (5/6)³ = {1 - (5/6) ** 3:.3f}')
    print('✓ richtig' if abs(k1 - 0.4213) < 0.01 else '✗ prüfen')

In [ ]:
# TODO K2: Geburtstagsproblem – Anteil der Gruppen à 23 Personen mit mindestens einem doppelten Geburtstag
gruppen = rng.integers(0, 365, size=(10_000, 23))
k2 = None   # Tipp: für jede Zeile prüfen, ob len(set(zeile)) < 23

# --- Check ---
if k2 is None:
    print('… noch offen')
else:
    print(f'simuliert: {k2:.3f} · gerechnet: 0.507')
    print('✓ richtig' if abs(k2 - 0.507) < 0.02 else '✗ prüfen')

In [ ]:
# TODO K3: Ziegenproblem – drei Türen, ein Auto. Sie wählen, der Moderator öffnet eine Ziegentür, Sie wechseln.
# Wer wechselt, gewinnt genau dann, wenn die erste Wahl falsch war. Simulieren Sie 10 000 Spiele.
auto = rng.integers(0, 3, size=10_000)
wahl = rng.integers(0, 3, size=10_000)
k3 = None   # Anteil der Spiele, die man durch Wechseln gewinnt

# --- Check ---
if k3 is None:
    print('… noch offen')
else:
    print(f'Wechseln gewinnt: {k3:.3f} · Bleiben gewinnt: {1 - k3:.3f}')
    print('✓ richtig' if abs(k3 - 2 / 3) < 0.02 else '✗ prüfen')

---
## Ohne Code: eine `describe()`-Ausgabe lesen

Monatsgehälter (brutto, €) in einer Abteilung mit 20 Beschäftigten:

| count | mean | std | min | 25% | 50% | 75% | max |
|---|---|---|---|---|---|---|---|
| 20 | 4 500 | 2 563 | 2 900 | 3 400 | 3 900 | 4 600 | 15 000 |

1. Welche Zahl beschreibt ein typisches Gehalt besser, und warum?
2. Wie breit ist die mittlere Hälfte der Gehälter?
3. Ist 15 000 € nach der Regel aus der Vorlesung ein Ausreißer?

*Ihre Antworten:*

---
## Zum Schluss: Restart & Run All

> Menü **Laufzeit → Sitzung neu starten und alle ausführen**. Läuft das Notebook danach ohne Fehler von oben bis unten
> durch, ist es fertig.

**Für die Besprechung um 18:05:** Ein Team zeigt seine drei Sätze aus Aufgabe 6, ein Team die Boxplots aus Aufgabe 4.

**Nächste Woche:** Ceará 13,8 %, São Paulo 4,5 %: Ist dieser Unterschied echt, oder könnte er Zufall sein?